# A4 | 15– Avaliar candidatos e registrar decisão

**Pessoa.** Consolidar a seleção **sem privilegiar apenas a maior métrica**.

**Saída:** ata, tabela de decisão e justificativa dos candidatos.
**Revisão:** B apresenta estabilidade/perfis; C apresenta a comparação supervisionada.

In [ ]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "minimas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
RESULTS = "outputs/tables/resultados_cv.csv"   # CSV de C: modelo, fold, pr_auc, roc_auc, brier
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A4")
# (esta etapa não precisa carregar a base)

Raiz do repositório: c:\Users\DUDU\Documents\GitHub\Projeto_Integrador
Pacote (src/protocolo): c:\Users\DUDU\Documents\GitHub\Projeto_Integrador
Config de decisões: c:\Users\DUDU\Documents\GitHub\Projeto_Integrador\configs\decisoes.yaml


## 1. Tabela comparável de candidatos
Média e variação entre folds, ganho sobre baseline e logística, Brier e alerta de complexidade sem ganho.

In [2]:
if Path(RESULTS).exists():
    res = pd.read_csv(RESULTS)
    v.a3_suspicious_metrics(res, cfg, rep)
    v.a4_decision_table(res, cfg, rep)
    display(pd.read_csv(Path(OUT) / "a4_tabela_decisao.csv"))
else:
    print("Resultados não encontrados:", RESULTS)

Resultados não encontrados: outputs/tables/resultados_cv.csv


## 2. Revisão por candidato (manual)
**Clustering:** estabilidade, tamanho mínimo, perfil e influência das ausências.
**Classificação:** média e variação entre folds, PR-AUC, ROC-AUC, Brier e calibração.

- [ ] Comparar modelo complexo com baseline e logística
- [ ] O ganho justifica a complexidade e a menor interpretabilidade?
- [ ] Revisar erros e limitações já visíveis
- [ ] Preencher `decisao` e `justificativa_descarte_ou_manutencao` em `a4_tabela_decisao.csv`

Busque por: *bias variance tradeoff model complexity official tutorial*, *discrimination versus calibration prediction model*, *cluster stability bootstrap resampling*, *permutation importance scikit-learn user guide*.

## 3. Decisões e ata de 17/10

In [3]:
E = "A4"
v.log_decision(OUT, "D-11", "Seleção de candidatos",
               "Shortlist por validade, estabilidade, calibração e interpretação — não só maior métrica", E,
               impacto="Candidatos para confirmação em 18/10")

SHORTLIST = ["(preencher) clustering: solução principal + alternativa", "(preencher) supervisionado: 1–2 modelos"]
DESCARTES = ["(preencher) candidato — justificativa padronizada"]
RISCOS = ["(preencher)"]
PENDENCIAS = ["Confirmação dos finalistas e congelamento em 18/10"]
v.write_ata(OUT, "Ata 17/10 — Fine Tuning", "2026-10-17",
            decisoes=["Busca encerrada ou finalizando; sem novas famílias de modelo"],
            pendencias=PENDENCIAS, riscos=RISCOS, shortlist=SHORTLIST, descartes=DESCARTES, rep=rep)
print(sorted(p.name for p in Path(OUT).glob("ata_*.md")))

['ata_17_10_fine_tuning.md', 'ata_gate_2_11_10.md', 'ata_gate_3_14_10.md']


## 4. População analítica congelada
A população analítica deve ficar congelada para a modelagem: o SHA-256 do arquivo precisa bater com o gravado na exportação.

In [4]:
caminho = v._analytical_path(cfg)
if caminho.exists():
    v.verificar_dataset_congelado(caminho, rep)
else:
    print("Dataset analítico ainda não gerado:", caminho)

[ok] A4 | Dataset congelado: dataset_supervisionado.parquet v1.0 inalterado desde a exportação


## 5. Fechamento

In [5]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))

Relatório: outputs\tables\pessoa_a\relatorio_A4.md


,etapa,verificação,status,detalhe
0,A4,Dataset congelado,OK,dataset_supervisionado.parquet v1.0 inalterado...


### Checklist de saída
- [ ] Tabela de decisão preenchida (decisão + justificativa por candidato)
- [ ] Ata fechada: busca encerrada ou finalizando, shortlist, descartes, riscos, pendências
- [ ] Critérios para abrir a validação final definidos
- [ ] Cada pessoa explica coorte, clustering, supervisionado e avaliação em nível conceitual